In [ ]:
import matplotlib.pyplot as plt
import base64
from io import BytesIO

def plot_generation_curve(df, model, task, column):
    """
    df: reg_df (generation-level)
    model: model name
    task: task name
    column: e.g. 'H_spatial_z' or 'H_fitness_z'
    """
    sub = df[(df["model"] == model) & (df["task"] == task)].sort_values("generation")

    plt.figure(figsize=(3.2, 1.8))
    plt.plot(sub["generation"], sub[column], marker='o', linewidth=1.2)
    plt.title(column, fontsize=8)
    plt.xlabel("gen", fontsize=7)
    plt.ylabel("", fontsize=7)
    plt.tight_layout()

    # encode as base64
    buf = BytesIO()
    plt.savefig(buf, format="png")
    plt.close()
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode("utf-8")
    return f'<img src="data:image/png;base64,{encoded}" width="260"/>'

In [ ]:
import matplotlib.pyplot as plt
import base64
from io import BytesIO

def plot_generation_curve(df, model, task, column):
    """
    df: reg_df (generation-level)
    model: model name
    task: task name
    column: e.g. 'H_spatial_z' or 'H_fitness_z'
    """
    sub = df[(df["model"] == model) & (df["task"] == task)].sort_values("generation")

    plt.figure(figsize=(3.2, 1.8))
    plt.plot(sub["generation"], sub[column], marker='o', linewidth=1.2)
    plt.title(column, fontsize=8)
    plt.xlabel("gen", fontsize=7)
    plt.ylabel("", fontsize=7)
    plt.tight_layout()

    # encode as base64
    buf = BytesIO()
    plt.savefig(buf, format="png")
    plt.close()
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode("utf-8")
    return f'<img src="data:image/png;base64,{encoded}" width="260"/>'


def make_pair_html_table(matched_pair_df, reg_df, output="matched_pairs.html"):

    html_rows = []
    pair_id = 1

    for idx, row in matched_pair_df.iterrows():
        task = row["task"]
        A, B = row["model_A"], row["model_B"]

        # --- plot images for A ---
        A_spatial = plot_generation_curve(reg_df, A, task, "H_spatial_z")
        A_fitness = plot_generation_curve(reg_df, A, task, "H_fitness_z")

        # --- plot images for B ---
        B_spatial = plot_generation_curve(reg_df, B, task, "H_spatial_z")
        B_fitness = plot_generation_curve(reg_df, B, task, "H_fitness_z")

        # --- A row ---
        row_A = f"""
        <tr>
            <td rowspan="2">{pair_id}</td>
            <td>{A}</td>
            <td>{row['zero_shot_A']:.3f}</td>
            <td>{row['final_A']:.3f}</td>
            <td>{A_spatial}</td>
            <td>{A_fitness}</td>
        </tr>
        """

        # --- B row ---
        row_B = f"""
        <tr>
            <td>{B}</td>
            <td>{row['zero_shot_B']:.3f}</td>
            <td>{row['final_B']:.3f}</td>
            <td>{B_spatial}</td>
            <td>{B_fitness}</td>
        </tr>
        """

        html_rows.append(row_A + row_B)
        pair_id += 1

    # --- Final HTML page ---
    html = f"""
    <html>
    <head>
        <style>
            table {{
                border-collapse: collapse;
                width: 100%;
            }}
            th, td {{
                border: 1px solid #ccc;
                padding: 6px;
                text-align: center;
                vertical-align: middle;
            }}
            th {{
                background-color: #eee;
            }}
        </style>
    </head>
    <body>
        <h2>Matched Model Pairs: Trajectory-Level Comparison</h2>
        <table>
            <tr>
                <th>Pair</th>
                <th>Model</th>
                <th>Zero-Shot</th>
                <th>Final</th>
                <th>H_spatial (per gen)</th>
                <th>H_fitness (per gen)</th>
            </tr>
            {''.join(html_rows)}
        </table>
    </body>
    </html>
    """

    with open(output, "w") as f:
        f.write(html)

    print(f"HTML file saved to {output}")

make_pair_html_table(matched_pair_df, reg_df)

In [ ]:
from collections import defaultdict
import pandas as pd
import numpy as np
from pathlib import Path


TASKS = {
    "Combinatorial Optimization": [
        ("TSP-30", "/Users/lievretre/Desktop/evo_eval_sheets/tsp30_raw.csv"),
        ("TSP-60", "/Users/lievretre/Desktop/evo_eval_sheets/tsp60_raw.csv"),
    ],
    "Symbolic Regression": [
        ("Oscillator-1", "/Users/lievretre/Desktop/evo_eval_sheets/oscillator1_raw.csv"),
        ("Oscillator-2", "/Users/lievretre/Desktop/evo_eval_sheets/oscillator2_raw.csv"),
    ],
    "Heuristic Design": [
        ("Bin-Packing-Weibull", "/Users/lievretre/Desktop/evo_eval_sheets/bin_packing_weibull_raw.csv"),
        ("Bin-Packing-OR3", "/Users/lievretre/Desktop/evo_eval_sheets/bin_packing_or3_raw.csv"),
    ],
    "Prompt Optimization": [
        ("Summarization", "/Users/lievretre/Desktop/evo_eval_sheets/promptopt_sum_raw.csv"),
        ("Simplication", "/Users/lievretre/Desktop/evo_eval_sheets/promptopt_sim_raw.csv"),
    ],
}

def rewrite_task_column_in_index_csvs(TASKS):
    """
    对 TASKS 字典中的每个 index-csv 文件进行处理：
    → 直接覆盖 csv 中的 task 列，让所有行的 task = task_name（来自 TASKS）
    → 不生成新文件，直接修改原 CSV
    """

    updated_paths = {}

    for category, items in TASKS.items():
        for task_name, csv_path in items:

            csv_path = Path(csv_path)
            if not csv_path.exists():
                print(f"[skip] file not found: {csv_path}")
                continue

            df = pd.read_csv(csv_path)

            # 强制覆盖 task 列（如果不存在则新增）
            df["task"] = task_name

            # 直接覆盖原文件
            df.to_csv(csv_path, index=False)

            updated_paths[(category, task_name)] = str(csv_path)

            print(f"[ok] updated: task column in {csv_path.name} → '{task_name}'")

    return updated_paths
    
def build_index_from_TASKS(TASKS):
    """
    把所有 task 对应的 index-csv 合并成一个统一的 index_df
    要求每个 csv 至少包含：model, csv_file_path
    如果没有 task 列，就用 task_name 填上
    """
    dfs = []

    for category, task_list in TASKS.items():
        for task_name, index_csv_path in task_list:
            p = Path(index_csv_path)
            if not p.exists():
                print(f"[skip] index csv not found: {p}")
                continue

            df = pd.read_csv(p)

            # 确保有 task 列
            if "task" not in df.columns:
                df["task"] = task_name
            else:
                # 如果原表里 task 列是空/混乱，也可以直接覆盖：
                # df["task"] = task_name
                pass

            # 确保有 csv_file_path 列（有些你可能叫 path / csv_path）
            if "csv_file_path" not in df.columns:
                for cand in ["csv_path", "path"]:
                    if cand in df.columns:
                        df = df.rename(columns={cand: "csv_file_path"})
                        break

            if "csv_file_path" not in df.columns:
                raise ValueError(f"{p} 里找不到 csv_file_path / csv_path / path 列")

            # 只保留我们后面要用的几列，多余的列保留也没坏处
            keep_cols = ["task", "model", "csv_file_path"]
            extra = [c for c in df.columns if c not in keep_cols]
            df = df[keep_cols + extra]

            dfs.append(df)

    if not dfs:
        raise ValueError("没有任何有效的 index csv 被加载，请检查 TASKS 路径。")

    index_df = pd.concat(dfs, ignore_index=True)
    return index_df

def build_task_model_panels(index_df):
    """
    index_df 必须包含:
        task
        model
        csv_file_path  (必须指向 generation-level 轨迹 CSV)
    
    返回:
        panels[task][model] = DataFrame( generation, fitness_norm, novelty_norm )
    """

    panels = defaultdict(dict)
    raw_novelty = defaultdict(list)

    for _, row in index_df.iterrows():

        task = row["task"]
        model = row["model"]
        process_csv_path = row["csv_file_path"]  # ★ 必须是轨迹文件路径

        process_csv_path = Path(process_csv_path)
        if not process_csv_path.exists():
            print(f"[skip] missing process csv: {process_csv_path}")
            continue

        # 读取 process 轨迹文件，而不是 index 文件
        df = pd.read_csv(process_csv_path)

        # 必须包含 generation-level 列
        required_cols = ["generation", "fitness_normed", "total_distance"]
        if not set(required_cols).issubset(df.columns):
            print(f"[skip] missing required columns in process csv: {process_csv_path}")
            continue

        df = df[required_cols].copy()
        df = df.dropna()

        df["generation"]    = pd.to_numeric(df["generation"], errors="coerce")
        df["fitness_norm"]  = pd.to_numeric(df["fitness_normed"], errors="coerce")
        df["novelty_raw"]   = pd.to_numeric(df["total_distance"], errors="coerce")
        df = df.dropna()

        if df.empty:
            continue

        # 收集 raw novelty 用于 task-level normalization
        raw_novelty[task].append(df["novelty_raw"])

        # 保存：按 task, model 分组
        if model not in panels[task]:
            panels[task][model] = df
        else:
            panels[task][model] = pd.concat([panels[task][model], df], ignore_index=True)

    # ========== TASK-wise 归一化 ==========
    norm_panels = defaultdict(dict)

    for task, model_dict in panels.items():

        vec = pd.concat(raw_novelty[task], ignore_index=True)
        lo, hi = np.nanpercentile(vec, [1, 99])
        den = hi - lo if hi > lo else 1.0

        for model, df in model_dict.items():
            df2 = df.copy()
            df2["novelty_norm"] = ((df2["novelty_raw"] - lo) / den).clip(0, 1)
            norm_panels[task][model] = df2

    return norm_panels
rewrite_task_column_in_index_csvs(TASKS)
index_df = build_index_from_TASKS(TASKS)
panels = build_task_model_panels(index_df)  

In [ ]:
import matplotlib
matplotlib.use("Agg")          # 顶部只设置一次
import matplotlib.pyplot as plt
import base64
from io import BytesIO
import numpy as np


def pareto_front_xy(x: np.ndarray, y: np.ndarray, direction: str = "ltr") -> np.ndarray:
    ok = np.isfinite(x) & np.isfinite(y)
    if not ok.any():
        return np.empty((0, 2))
    xv = x[ok]; yv = y[ok]
    if direction == "rtl":
        order = np.argsort(xv)[::-1]          # x 降序
        xv = xv[order]; yv = yv[order]
        best = -np.inf
        pts = []
        for xi, yi in zip(xv, yv):
            if yi > best:
                pts.append((xi, yi)); best = yi
        return np.array(pts)
    else:
        order = np.argsort(xv)                # x 升序
        xv = xv[order]; yv = yv[order]
        best = -np.inf
        pts = []
        for xi, yi in zip(xv, yv):
            if yi > best:
                pts.append((xi, yi)); best = yi
        return np.array(pts)


def plot_scatter_single_df(task, model, df):
    """df 需要包含: generation, fitness_norm, novelty_norm"""
    if df is None or df.empty:
        return ""

    plt.figure(figsize=(3.2, 3))
    ax = plt.gca()

    # 简单按 generation 上色（0–1 范围）
    gens = df["generation"].to_numpy(float)
    if gens.max() > gens.min():
        g_norm = (gens - gens.min()) / (gens.max() - gens.min())
    else:
        g_norm = np.zeros_like(gens)
    cmap = plt.get_cmap("viridis")
    colors = cmap(g_norm)

    ax.scatter(
        df["fitness_norm"], df["novelty_norm"],
        c=colors, s=15, edgecolor="white", linewidth=0.3, alpha=0.95
    )

    # median 轨迹
    med = df.groupby("generation", as_index=False).median(numeric_only=True)
    if not med.empty:
        ax.plot(med["fitness_norm"], med["novelty_norm"], color="orange", lw=1.6)

    # Pareto front
    pts = pareto_front_xy(df["fitness_norm"].to_numpy(float),
                          df["novelty_norm"].to_numpy(float),
                          direction="rtl")
    if len(pts) > 0:
        ax.plot(pts[:, 0], pts[:, 1], color="steelblue", lw=1.5)

    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, color="#eee")
    ax.set_title(f"{task} | {model}", fontsize=9)

    buf = BytesIO()
    plt.tight_layout()
    plt.savefig(buf, format="png", dpi=120)
    plt.close()

    buf.seek(0)                                   # ★★ 关键：重置指针
    img_bytes = buf.read()
    return base64.b64encode(img_bytes).decode("utf-8")


def precompute_scatter_cache(panels):
    cache = {}

    for task, model_dict in panels.items():
        for model, df in model_dict.items():
            img = plot_scatter_single_df(task, model, df)
            cache[(task, model)] = img

    return cache

In [ ]:
def check_cache_validity(scatter_cache):
    empty = []
    valid = 0

    for key, img in scatter_cache.items():
        if isinstance(img, str) and len(img) > 500:
            valid += 1
        else:
            empty.append((key, img))

    print("Valid images:", valid)
    print("Empty images:", len(empty))

    if empty:
        print("\n--- Empty entries detail ---")
        for k, v in empty[:10]:
            print("Key:", k, "| value len:", 0 if v is None else len(v))
scatter_cache = precompute_scatter_cache(panels)
check_cache_validity(scatter_cache)

In [ ]:
all_mds_cache = {}

# 合并 TSP
for key, img in mds_cache.items():
    all_mds_cache[key] = img

# 合并 SR
for key, img in sr_mds_cache.items():
    all_mds_cache[key] = img

# 合并 OBP
for key, img in OBP_MDS_cache.items():
    all_mds_cache[key] = img

# 合并 Promptopt
for key, img in PROMPTOPT_MDS_cache.items():
    all_mds_cache[key] = img

In [ ]:
import re

def canonicalize_model(raw: str) -> str:
    """
    将各种不规则模型名映射到 reg_df 使用的 canonical 模型名称。
    （structure 不变，只替换 model 部分）
    """

    s = str(raw)

    # 去掉尾部 _np2_nc5_simple、_simple 等实验性后缀
    s = re.sub(r'_np\d+_nc\d+_simple$', '', s)
    s = re.sub(r'_simple$', '', s)
    s = s.replace(":free", "")

    # 常见前缀：openai_gpt → gpt、meta-llama_llama → llama
    s = s.replace("openai_", "")
    s = s.replace("vertex_ai_", "")
    s = s.replace("meta-llama_", "")
    s = s.replace("google_", "")
    s = s.replace("mistralai_", "")
    s = s.replace("deepseek_", "")
    
    # gemma 的特殊情况
    s = s.replace("gemma-3n-e4b-it", "gemma-3n-4b")

    # mistral special
    s = s.replace("mistral-small-3.2-24b-instruct", "mistral-24b-instruct")
    s = s.replace("mistral-7b-instruct-v0.3", "mistral-7b-instruct")
    s = s.replace("magistral-small-2506", "mistral-magistral-small")

    # 去掉双重前缀，如 meta-llama_llama-3.1 → llama-3.1
    s = re.sub(r'.*?(llama-[0-9].*)', r'\1', s)

    # 最常见 openai 模型
    s = s.replace("gpt-4o-mini", "gpt-4o-mini")
    s = s.replace("gpt-4o", "gpt-4o")
    s = s.replace("gpt-3.5-turbo", "gpt-3.5-turbo")

    # vertex/flash/pro

    s = s.replace("gemini-1.5", "gemini-1.5-pro")
    s = s.replace("gemini-1.5-pro-flash", "gemini-1.5-flash")
    s = s.replace("gemini-1.5-pro-pro", "gemini-1.5-pro")
    # deepseek
    s = s.replace("deepseek-chat-v3-0324", "deepseek-v3-chat")

    s = s.replace("binpacking_or3", "Bin-Packing-OR3")
    s = s.replace("binpacking_weibull", "Bin-Packing-Weibull")
    
    
    # 去掉首尾空格
    return s.strip()

fixed_cache = {}

for (task, model), img in all_mds_cache.items():
    model_clean = canonicalize_model(model)
    fixed_cache[(task, model_clean)] = img
fixed_cache_new = {}  
for (task, model), img in fixed_cache.items():
    task_clean = canonicalize_model(task)
    fixed_cache_new[(task_clean, model)] = img

In [ ]:
set([i for (i,j) in fixed_cache_new.keys()])
set(reg_df["task"].unique())

In [ ]:
import matplotlib.pyplot as plt
import base64
from io import BytesIO
import pandas as pd

def plot_curve_base64(sub, column):
    """绘制图像并编码为 base64"""
    plt.figure(figsize=(3, 1.6))
    plt.plot(sub["generation"], sub[column], marker="o", linewidth=1)
    plt.title(column, fontsize=8)
    plt.xlabel("gen", fontsize=7)
    plt.ylabel("", fontsize=7)
    plt.tight_layout()

    buf = BytesIO()
    plt.savefig(buf, format="png")
    plt.close()
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")


def make_interactive_pair_html(matched_pair_df, reg_df, output="matched_pairs_interactive.html"):

    html_rows = []
    pair_id = 1

    for _, row in matched_pair_df.iterrows():

        task = row["task"]
        A, B = row["model_A"], row["model_B"]

        # model A data
        subA = reg_df[(reg_df["task"] == task) & (reg_df["model"] == A)].sort_values("generation")
        A_spatial_img = plot_curve_base64(subA, "H_spatial")
        A_fitness_img = plot_curve_base64(subA, "H_fitness")
        A_nwf_img = plot_curve_base64(subA, "nwf_t_mean")

        # ⭐⭐ 关键修改：加入散点缓存 ⭐⭐
        A_scatter = scatter_cache.get((task, A), None)
        if A_scatter is None:
            A_scatter_html = "N/A"
        else:
            A_scatter_html = f'<img src="data:image/png;base64,{A_scatter}" width="260">'
        A_mds_img = fixed_cache_new.get((task, A), None)
        if A_mds_img is None:
            A_mds_html = "N/A"
        else:
            A_mds_html = f'<img src="data:image/png;base64,{A_mds_img}" width="260">'
        row_A = f"""
        <tr>
            <td>{pair_id}</td>
            <td>{task}</td>
            <td>{A}</td>
            <td>{row['zero_shot_A']:.3f}</td>
            <td>{row['final_A']:.3f}</td>
            <td><img src="data:image/png;base64,{A_spatial_img}" width="250"></td>
            <td><img src="data:image/png;base64,{A_fitness_img}" width="250"></td>
            <td><img src="data:image/png;base64,{A_nwf_img}" width="250"></td>
            <td>{A_scatter_html}</td>
            <td>{A_mds_html}</td>
        </tr>
        """

        # model B data
        subB = reg_df[(reg_df["task"] == task) & (reg_df["model"] == B)].sort_values("generation")
        B_spatial_img = plot_curve_base64(subB, "H_spatial")
        B_fitness_img = plot_curve_base64(subB, "H_fitness")
        B_nwf_img = plot_curve_base64(subB, "nwf_t_mean")
        # ⭐⭐ 加入散点缓存 ⭐⭐
        B_scatter = scatter_cache.get((task, B), None)
        if B_scatter is None:
            B_scatter_html = "N/A"
        else:
            B_scatter_html = f'<img src="data:image/png;base64,{B_scatter}" width="260">'
        B_mds_img = fixed_cache_new.get((task, B), None)
        if B_mds_img is None:
            B_mds_html = "N/A"
        else:
            B_mds_html = f'<img src="data:image/png;base64,{B_mds_img}" width="260">'
        row_B = f"""
        <tr>
            <td>{pair_id}</td>
            <td>{task}</td>
            <td>{B}</td>
            <td>{row['zero_shot_B']:.3f}</td>
            <td>{row['final_B']:.3f}</td>
            <td><img src="data:image/png;base64,{B_spatial_img}" width="250"></td>
            <td><img src="data:image/png;base64,{B_fitness_img}" width="250"></td>
            <td><img src="data:image/png;base64,{B_nwf_img}" width="250"></td>
            <td>{B_scatter_html}</td>
            <td>{B_mds_html}</td>
        </tr>
        """

        html_rows.append(row_A)
        html_rows.append(row_B)

        pair_id += 1

    # ------------------ HTML Template with DataTables ------------------
    html = f"""
    <html>
    <head>
        <meta charset="UTF-8">
        <title>Matched Pairs Table</title>

        <!-- jQuery & DataTables -->
        <script src="https://code.jquery.com/jquery-3.5.1.js"></script>
        <link rel="stylesheet" href="https://cdn.datatables.net/1.13.1/css/jquery.dataTables.min.css">
        <script src="https://cdn.datatables.net/1.13.1/js/jquery.dataTables.min.js"></script>

        <style>
            table {{
                width: 100%;
                border-collapse: collapse;
            }}
            th, td {{
                border: 1px solid #ccc;
                padding: 4px;
                text-align: center;
            }}
            th {{
                background-color: #f0f0f0;
            }}
        </style>

        <script>
        $(document).ready(function() {{
            $('#pair_table').DataTable({{
                "paging": true,
                "pageLength": 20,
                "order": [],
                "columnDefs": [
                    {{ "orderable": false, "targets": [5, 6] }} // 图片列不排序
                ]
            }});
        }});
        </script>
    </head>

    <body>
        <h2>Matched Model Pairs (Trajectory Comparison)</h2>

        <table id="pair_table" class="display">
            <thead>
                <tr>
                    <th>Matched Pair ID</th>
                    <th>Task</th>
                    <th>Model</th>
                    <th>Zero-Shot</th>
                    <th>Best Final</th>
                    <th>H_spatial</th>
                    <th>H_fitness</th>
                    <th>NWF_t_mean</th>
                    <th>Fit-Nov Scatter</th>
                    <th>MDS Plot</th>
                </tr>
            </thead>
            <tbody>
                {''.join(html_rows)}
            </tbody>
        </table>

    </body>
    </html>
    """

    with open(output, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"Interactive HTML saved to: {output}")
make_interactive_pair_html(matched_pair_df, reg_df)

In [ ]:
import os, re, json, math, glob
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from sklearn.manifold import MDS


# MDS 超参
MDS_MAX_POINTS   = 4000      # 超过则抽样学平面 + OOS 插值
PER_BUCKET       = 60        # (model, generation) 每桶上限（用于抽样）
MDS_KW = dict(
    n_components=2,
    dissimilarity="precomputed",
    n_init=1,
    max_iter=300,
    eps=1e-3,
    random_state=42,
    verbose=1,
)

# OOS 插值：KNN & 权重
OOS_K = 8
OOS_P = 2.0     # 权重=1/(d+1e-8)^P
RNG = np.random.default_rng(42)

def build_tsp_mds_input(raw_csv_path, task_name):
    """
    输入一个 raw TSP CSV，输出：
    df_all: 包含 generation, fitness(normed), tour(list[int]), model
    """

    df = pd.read_csv(raw_csv_path)
    
    # 检查必须字段
    required = ["generation", "fitness_normed", "genome", "model"]
    miss = [c for c in required if c not in df.columns]
    if miss:
        print(f"[skip] missing {miss} in {raw_csv_path}")
        return None

    # 转换 tour
    def parse_tour(x):
        import ast
        try:
            v = ast.literal_eval(x)
            return [int(t) for t in v]
        except:
            toks = re.findall(r"\d+", x)
            return [int(t) for t in toks]

    df["tour"] = df["genome"].apply(parse_tour)
    df["fitness"] = df["fitness_normed"]

    # 生成任务层 dataframe
    out = df[["generation", "fitness", "tour", "model"]].copy()
    out["task"] = task_name

    return out

def find_col(df: pd.DataFrame, cands: List[str], required: bool=True) -> Optional[str]:
    for c in cands:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Required column not found. Tried: {cands}")
    return None

def load_index(index_path: str) -> pd.DataFrame:
    idx = pd.read_csv(index_path)
    path_col = find_col(idx, PATH_COL_CANDS, required=True)
    keep = [path_col]
    # 把 index 里跟模型有关的列也带上（用于兜底）
    for c in MODEL_COL_CANDS:
        if c in idx.columns: keep.append(c)
    out = idx[keep].copy()
    out.rename(columns={path_col: "csv_file_path"}, inplace=True)
    return out

def read_csv_maybe(path: str) -> Optional[pd.DataFrame]:
    p = Path(path)
    if not p.exists():
        print(f"[warn] missing CSV: {p}")
        return None
    try:
        return pd.read_csv(p)
    except Exception as e:
        print(f"[warn] read failed: {p} -> {e}")
        return None

def robust_minmax(x: np.ndarray, q=(1,99)):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return x
    lo, hi = np.nanpercentile(x, q)
    den = (hi-lo) if hi>lo else 1.0
    y = (x - lo) / den
    return np.clip(y, 0, 1)
    
def coerce_tour(x):
    """把 tour 解析成 list[int]"""
    if isinstance(x, list): return [int(v) for v in x]
    if isinstance(x, np.ndarray): return [int(v) for v in x.tolist()]
    if isinstance(x, str):
        try:
            v = ast.literal_eval(x)
            return [int(t) for t in (v.tolist() if isinstance(v, np.ndarray) else list(v))]
        except Exception:
            # 兜底：逗号/空格切
            toks = re.findall(r"\d+", x)
            return [int(t) for t in toks]
    raise ValueError(f"Unsupported tour type: {type(x)}")

def edge_indexer(n_cities: int):
    """
    返回：map[(i,j)] -> col_id  (i<j)
    用于把无向边映射到 0..E-1 的列索引
    """
    idx = {}
    col = 0
    for i in range(n_cities):
        for j in range(i+1, n_cities):
            idx[(i,j)] = col
            col += 1
    return idx, col  # col == E = n*(n-1)/2

def tours_to_incidence(tours, n_cities: int, edge_map):
    """
    tours: List[List[int]]
    返回 A: uint8 矩阵，形状 (S, E)，每行有 n_cities 个 1（闭环）
    """
    S = len(tours)
    E = len(edge_map)
    A = np.zeros((S, E), dtype=np.uint8)
    for r, t in enumerate(tours):
        m = len(t)
        # 闭环
        for k in range(m):
            a = int(t[k]); b = int(t[(k+1) % m])
            if a == b: 
                continue
            if a > b: 
                a, b = b, a
            A[r, edge_map[(a,b)]] = 1
    return A

def distances_on_fit(A_fit: np.ndarray, n_edges: int) -> np.ndarray:
    """
    A_fit: (m, E) uint8
    返回 D_fit: (m, m) float32，D = 1 - (|E∩| / n_edges)
    """
    # 交集大小 = 点积；用 int32/float32 计算
    # 注意：A_fit 是 0/1，A_fit @ A_fit.T 非常快（BLAS）
    inter = (A_fit @ A_fit.T).astype(np.float32)
    D = 1.0 - inter / float(n_edges)
    # 数值安全
    np.fill_diagonal(D, 0.0)
    return D
    
def distance_one_to_many_edges(Ei: frozenset, E_list: List[frozenset], n_edges: int) -> np.ndarray:
    """给一个边集，计算到一批边集的边差距离"""
    out = np.empty(len(E_list), dtype=np.float32)
    for k, Ej in enumerate(E_list):
        inter = len(Ei.intersection(Ej))
        out[k] = 1.0 - (inter / max(1, n_edges))
    return out

def robust_minmax(x: np.ndarray, q=(1,99)):
    lo, hi = np.nanpercentile(x, q)
    den = (hi-lo) if hi>lo else 1.0
    return np.clip((x-lo)/den, 0, 1)

def stratified_sample_idx(df: pd.DataFrame, per_bucket=PER_BUCKET, total_cap=MDS_MAX_POINTS) -> np.ndarray:
    parts = []
    for (m, g), sub in df.groupby(["model","generation"], sort=False):
        if len(sub) > per_bucket:
            parts.append(sub.sample(per_bucket, random_state=int(RNG.integers(1<<31))))
        else:
            parts.append(sub)
    out = pd.concat(parts, ignore_index=False)
    if len(out) > total_cap:
        out = out.sample(total_cap, random_state=42)
    return out.index.to_numpy()

def oos_place_by_blocks(A_all, fit_idx, Y_fit, n_edges, k=8, p=2.0, block=4000):
    """
    A_all: (N, E)  全体 0/1 发生矩阵
    fit_idx: (m,)  抽样索引
    Y_fit: (m,2)   MDS 基点坐标
    返回：Y: (N,2)
    """
    N = A_all.shape[0]
    m = len(fit_idx)
    Y = np.zeros((N, 2), dtype=np.float32)
    Y[fit_idx] = Y_fit

    not_mask = np.ones(N, dtype=bool)
    not_mask[fit_idx] = False
    q_idx = np.where(not_mask)[0]
    if len(q_idx) == 0:
        return Y

    A_fit = A_all[fit_idx]                   # (m, E)
    A_fitT = A_fit.T.astype(np.uint8, copy=False)

    for s in range(0, len(q_idx), block):
        sl = q_idx[s: s+block]               # 这一块查询
        Aq  = A_all[sl]                      # (b, E)
        # 交集数 = Aq @ A_fit^T
        inter = (Aq @ A_fitT).astype(np.float32)   # (b, m)
        Dq = 1.0 - inter / float(n_edges)         # (b, m)

        # 取每行的 k 个最近邻（最小距离）
        if k < m:
            # argpartition 更快
            nn = np.argpartition(Dq, kth=k, axis=1)[:, :k]   # (b, k)
            rows = np.arange(len(sl))[:, None]
            dnn = Dq[rows, nn]                               # (b, k)
        else:
            nn = np.tile(np.arange(m), (len(sl),1))
            dnn = Dq

        w = 1.0 / (dnn + 1e-8) ** p
        w = w / w.sum(axis=1, keepdims=True)                 # (b, k)
        # 加权平均坐标
        Y[sl] = (w[..., None] * Y_fit[nn]).sum(axis=1).astype(np.float32)

    return Y
def build_tsp_task_mds_panels():
    tasks = {
        "TSP-30": "/Users/lievretre/Desktop/evo_eval_sheets/tsp30_normed.csv",
        "TSP-60": "/Users/lievretre/Desktop/evo_eval_sheets/tsp60_normed.csv",
    }

    mds_panels = {}

    for task_name, path in tasks.items():
        df_all = build_tsp_mds_input(path, task_name)
        if df_all is None:
            continue

        print(f"[MDS] {task_name}: {len(df_all)} rows loaded")

        # === 1) 准备 tours → incidence matrix ===
        tours = df_all["tour"].tolist()
        n_cities = len(tours[0])

        edge_map, E = edge_indexer(n_cities)
        A_all = tours_to_incidence(tours, n_cities, edge_map)

        # === 2) 抽样 fit set ===
        if len(df_all) <= 4000:
            fit_idx = np.arange(len(df_all))
        else:
            fit_idx = stratified_sample_idx(df_all, per_bucket=60, total_cap=4000)

        A_fit = A_all[fit_idx]
        n_edges = n_cities

        D_fit = distances_on_fit(A_fit, n_edges)

        # === 3) MDS ===
        mds = MDS(
            n_components=2,
            dissimilarity="precomputed",
            random_state=42,
            max_iter=300,
            n_init=1,
            eps=1e-3
        )
        Y_fit = mds.fit_transform(D_fit)

        # === 4) OOS ===
        Y = oos_place_by_blocks(A_all, fit_idx, Y_fit, n_edges,
                                k=8, p=2.0, block=5000)

        df_all["mds_x"] = Y[:, 0]
        df_all["mds_y"] = Y[:, 1]

        # === 5) 分模型 ===
        mds_panels[task_name] = {}
        for model, dfm in df_all.groupby("model"):
            mds_panels[task_name][model] = dfm.reset_index(drop=True)

    return mds_panels

In [ ]:
import re, ast
from pathlib import Path

# 这些候选可以直接复用你之前 MDS 脚本里的
GEN_COL_CANDS   = ["generation", "gen", "g"]
FITN_COL_CANDS  = ["fitness_normed", "fitness", "score"]
TOUR_COL_CANDS  = ["genome", "path", "perm", "route", "cycle"]
TYPE_COL_CANDS  = ["type", "sample_type", "role"]

def build_tsp_task_big_df(index_df, child_only=False):
    """
    从全局 index_df 里，提取 TSP-30 / TSP-60 的所有轨迹行：
    返回:
        big_by_task["TSP-30"] = df_all
        big_by_task["TSP-60"] = df_all
    每个 df_all 至少包含: generation, fitness, tour(list[int]), model, task
    """
    big_by_task = {}

    for task_name in ["TSP-30", "TSP-60"]:
        sub_idx = index_df[index_df["task"] == task_name].copy()
        if sub_idx.empty:
            print(f"[MDS] no rows in index_df for {task_name}")
            continue

        rows = []

        for _, row in sub_idx.iterrows():
            model = row["model"]
            csv_path = Path(str(row["csv_file_path"]).strip())

            if not csv_path.exists():
                print(f"[MDS] [skip] missing process csv: {csv_path}")
                continue

            try:
                df_raw = pd.read_csv(csv_path)
            except Exception as e:
                print(f"[MDS] [skip] read failed: {csv_path} -> {e}")
                continue

            # 自动找列
            try:
                gen_col  = find_col(df_raw, GEN_COL_CANDS, required=True)
                fit_col  = find_col(df_raw, FITN_COL_CANDS, required=True)
                tour_col = find_col(df_raw, TOUR_COL_CANDS, required=True)
                type_col = find_col(df_raw, TYPE_COL_CANDS, required=False)
            except ValueError as e:
                print(f"[MDS] [skip] {csv_path}: {e}")
                continue

            df_sub = pd.DataFrame({
                "generation": pd.to_numeric(df_raw[gen_col], errors="coerce"),
                "fitness":   pd.to_numeric(df_raw[fit_col], errors="coerce"),
                "tour_raw":  df_raw[tour_col],
            })

            if type_col and type_col in df_raw.columns:
                df_sub["type"] = df_raw[type_col].astype(str)
            else:
                df_sub["type"] = "unknown"

            df_sub = df_sub.dropna(subset=["generation", "fitness", "tour_raw"])

            # 只要 child（可选）
            if child_only:
                df_sub = df_sub[df_sub["type"].str.lower().eq("child")]
                if df_sub.empty:
                    continue

            # tour 解析
            df_sub["tour"] = df_sub["tour_raw"].apply(coerce_tour)

            # 写入 model & task
            df_sub["model"] = str(model)
            df_sub["task"]  = task_name

            rows.append(df_sub)

        if not rows:
            print(f"[MDS] [warn] no usable rows for {task_name}")
            continue

        big = pd.concat(rows, ignore_index=True)
        big_by_task[task_name] = big
        print(f"[MDS] {task_name}: collected {len(big)} rows, models={big['model'].nunique()}")

    return big_by_task

from sklearn.manifold import MDS
import numpy as np

def build_tsp_mds_panels_from_index(index_df, child_only=False):
    """
    使用 index_df（已经有 task / model / csv_file_path）
    返回:
        mds_panels[task][model] = df
    每个 df 至少包含: generation, fitness_normed, mds_x, mds_y, model, task
    """
    big_by_task = build_tsp_task_big_df(index_df, child_only=child_only)
    mds_panels = {}

    for task_name, big in big_by_task.items():
        big = big.reset_index(drop=True)

        # === 1) tours -> incidence matrix ===
        tours = big["tour"].tolist()
        n_cities = len(tours[0])

        edge_map, E = edge_indexer(n_cities)
        A_all = tours_to_incidence(tours, n_cities, edge_map)

        # === 2) 选取 fit 集合（避免全量 MDS 爆炸） ===
        if len(big) <= 4000:
            fit_idx = np.arange(len(big))
        else:
            fit_idx = stratified_sample_idx(big, per_bucket=60, total_cap=4000)

        A_fit = A_all[fit_idx]
        n_edges = n_cities

        D_fit = distances_on_fit(A_fit, n_edges)

        print(f"[MDS] {task_name}: fitting on {len(fit_idx)} points (of {len(big)})")

        # === 3) MDS ===
        mds = MDS(
            n_components=2,
            dissimilarity="precomputed",
            n_init=1,
            max_iter=300,
            eps=1e-3,
            random_state=42,
            verbose=1,
        )
        Y_fit = mds.fit_transform(D_fit)

        # === 4) OOS 插值 ===
        Y = oos_place_by_blocks(A_all, fit_idx, Y_fit, n_edges,
                                k=8, p=2.0, block=6000)

        big["mds_x"] = Y[:, 0]
        big["mds_y"] = Y[:, 1]

        # 用 fitness 做一个 0-1 的归一化，给 marker size 用
        big["fitness_normed"] = robust_minmax(big["fitness"].to_numpy(), q=(1, 99))

        # === 5) 分 model 存入 mds_panels ===
        mds_panels[task_name] = {}
        for model, dfm in big.groupby("model"):
            mds_panels[task_name][model] = dfm[[
                "generation", "fitness_normed", "mds_x", "mds_y", "model", "task"
            ]].reset_index(drop=True)

    return mds_panels

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from io import BytesIO
import base64

def plot_mds_single_df(task, model, df):
    plt.figure(figsize=(3.2, 3))
    ax = plt.gca()

    # 颜色 = generation
    gen = df["generation"].to_numpy()
    gnorm = Normalize(vmin=gen.min(), vmax=gen.max())
    cmap = plt.get_cmap("viridis")
    colors = cmap(gnorm(gen))

    # 尺寸 = fitness_normed
    sizes = 10 + 80 * df["fitness_normed"].to_numpy()

    ax.scatter(df["mds_x"], df["mds_y"],
               s=sizes, c=colors,
               edgecolor="white", linewidth=0.35, alpha=0.92)

    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{task} | {model}", fontsize=9)

    buf = BytesIO()
    plt.tight_layout()
    plt.savefig(buf, format="png", dpi=120)
    plt.close()
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")

def precompute_mds_cache(mds_panels):
    cache = {}
    for task, model_dict in mds_panels.items():
        for model, df in model_dict.items():
            img = plot_mds_single_df(task, model, df)
            cache[(task, model)] = img
    return cache

# === 实际跑 ===
mds_panels = build_tsp_mds_panels_from_index(index_df, child_only=False)
mds_cache  = precompute_mds_cache(mds_panels)

# 简单 sanity check
valid = sum(1 for v in mds_cache.values() if isinstance(v, str) and len(v) > 0)
print("MDS valid images:", valid, " / ", len(mds_cache))

In [ ]:
# ================================================================
# Symbolic Regression — 构建 (task, model) 面板，用于 HTML 中嵌入图像
# ================================================================
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.manifold import MDS
from matplotlib.colors import Normalize
import matplotlib.pyplot as plt
import base64
from io import BytesIO

# MDS 超参（与 TSP 对齐）
MDS_MAX_POINTS   = 4000      # 超过则抽样学平面 + OOS 插值
PER_BUCKET       = 60        # (model, generation) 每桶上限（用于抽样）
MDS_KW = dict(
    n_components=2,
    dissimilarity="precomputed",
    n_init=1,
    max_iter=300,
    eps=1e-3,
    random_state=42,
    verbose=1,
)

# OOS 插值：KNN & 权重
OOS_K = 8
OOS_P = 2.0     # 权重=1/(d+1e-8)^P
RNG = np.random.default_rng(42)

# 预计算距离矩阵缓存（可选）
USE_PRECOMPUTED = True

# ----- 复用你之前的工具 -----
def parse_embedding(x):
    if isinstance(x, (list, np.ndarray)):
        arr = np.asarray(x, dtype=np.float32)
        return arr if arr.ndim == 1 else None
    if isinstance(x, str):
        try:
            arr = np.asarray(json.loads(x), dtype=np.float32)
            return arr if arr.ndim == 1 else None
        except:
            return None
    return None

def robust_minmax(x, q=(1,99)):
    lo, hi = np.nanpercentile(x, q)
    den = hi-lo if hi > lo else 1
    return np.clip((x-lo)/den, 0, 1)

def l2_normalize_rows(A, eps=1e-12):
    n = np.linalg.norm(A, axis=1, keepdims=True)
    n = np.where(n > eps, n, 1)
    return A / n

def cosine_D_fit(A_fit):
    S = np.clip(A_fit @ A_fit.T, -1, 1)
    D = 1 - S
    np.fill_diagonal(D, 0)
    return D

def cosine_D_query_to_fit(Aq, Af, block=4000):
    b, m = Aq.shape[0], Af.shape[0]
    out = np.zeros((b, m), np.float32)
    for s in range(0, b, block):
        sl = slice(s, min(b, s+block))
        S = np.clip(Aq[sl] @ Af.T, -1, 1)
        out[sl] = 1 - S
    return out

def oos_place_by_knn_cos(A_all, fit_idx, Y_fit, k=8, p=2, block=5000):
    N = A_all.shape[0]
    Y = np.zeros((N,2), np.float32)
    Y[fit_idx] = Y_fit

    mask = np.ones(N, bool)
    mask[fit_idx] = False
    qidx = np.where(mask)[0]

    Af = l2_normalize_rows(A_all[fit_idx])

    for s in range(0, len(qidx), block):
        ids = qidx[s:s+block]
        Aq = l2_normalize_rows(A_all[ids])
        Dq = cosine_D_query_to_fit(Aq, Af)

        nn = np.argpartition(Dq, k, axis=1)[:, :k]
        rows = np.arange(len(ids))[:,None]
        dnn = Dq[rows, nn]

        w = 1/(dnn + 1e-8)**p
        w = w / w.sum(axis=1, keepdims=True)
        Y[ids] = (w[...,None] * Y_fit[nn]).sum(axis=1)
    return Y
def build_sr_mds_panels_from_index(index_df):
    """
    输入: index_df（必须包含：task, model, csv_file_path）
    输出: sr_panels[task][model] = df_with_mds
    """
    sr_panels = {}

    # 只挑 Symbolic Regression
    SR_TASKS = ["Oscillator-1", "Oscillator-2"]
    sr_index = index_df[index_df["task"].isin(SR_TASKS)]

    for task in SR_TASKS:
        sub_idx = sr_index[sr_index["task"] == task]
        if sub_idx.empty:
            continue

        rows = []

        for _, row in sub_idx.iterrows():
            csv_path = Path(row["csv_file_path"])
            if not csv_path.exists():
                continue

            df = pd.read_csv(csv_path)
            if "embedding" not in df.columns:
                continue

            df2 = pd.DataFrame({
                "generation": df["generation"],
                "fitness_raw": df["fitness_normed"] if "fitness_normed" in df else df["fitness"],
                "emb": df["embedding"].apply(parse_embedding),
                "model": row["model"]
            }).dropna()

            rows.append(df2)

        if not rows:
            continue

        big = pd.concat(rows, ignore_index=True)

        # ==== 构造 embedding 矩阵 ====
        lens = big["emb"].apply(lambda x: len(x))
        mode_len = lens.mode().iloc[0]
        big = big[lens == mode_len].reset_index(drop=True)

        A = np.stack(big["emb"].to_numpy()).astype(np.float32)
        N, P = A.shape

        # ==== 选择 fit 子集 ====
        if N <= 4000:
            fit_idx = np.arange(N)
        else:
            fit_idx = np.random.choice(N, 4000, replace=False)

        A_fit = l2_normalize_rows(A[fit_idx])
        D_fit = cosine_D_fit(A_fit)

        # ==== MDS ====
        mds = MDS(
            n_components=2,
            dissimilarity="precomputed",
            n_init=1,
            max_iter=300,
            eps=1e-3,
            random_state=42
        )
        Y_fit = mds.fit_transform(D_fit)

        # ==== OOS ====
        Y = oos_place_by_knn_cos(A, fit_idx, Y_fit)

        big["mds_x"] = Y[:,0]
        big["mds_y"] = Y[:,1]
        big["fitness_normed"] = robust_minmax(big["fitness_raw"].to_numpy())

        # 分 model
        sr_panels[task] = {}
        for model, dfm in big.groupby("model"):
            sr_panels[task][model] = dfm[[
                "generation","fitness_normed","mds_x","mds_y","model"
            ]].reset_index(drop=True)

    return sr_panels

In [ ]:
def plot_sr_mds_single(task, model, df):
    plt.figure(figsize=(3.2,3))
    ax = plt.gca()

    gen = df["generation"].to_numpy()
    cmap = plt.get_cmap("viridis")
    norm = Normalize(vmin=gen.min(), vmax=gen.max())
    colors = cmap(norm(gen))

    sizes = 10 + 80 * df["fitness_normed"].to_numpy()

    ax.scatter(df["mds_x"], df["mds_y"],
               s=sizes, c=colors,
               edgecolor="white", linewidth=0.3, alpha=0.95)

    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{task} | {model}", fontsize=9)

    buf = BytesIO()
    plt.tight_layout()
    plt.savefig(buf, format="png", dpi=120)
    plt.close()
    buf.seek(0)

    return base64.b64encode(buf.read()).decode("utf-8")


def precompute_sr_mds_cache(sr_panels):
    cache = {}
    for task, model_dict in sr_panels.items():
        for model, df in model_dict.items():
            cache[(task, model)] = plot_sr_mds_single(task, model, df)
    return cache
sr_mds_panels = build_sr_mds_panels_from_index(index_df)
sr_mds_cache  = precompute_sr_mds_cache(sr_mds_panels)

print("SR MDS cache entries:", len(sr_mds_cache))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from sklearn.manifold import MDS
import base64
from io import BytesIO
import glob, json, math, re
from collections import defaultdict

# 列名候选（自动探测）
PATH_COL_CANDS   = ["csv_file_path", "csv_path", "path"]
GEN_COL_CANDS    = ["generation", "gen", "g"]
TYPE_COL_CANDS   = ["type", "sample_type", "role"]
FITN_COL_CANDS   = ["fitness_normed", "fitness", "score", "objective", "loss"]
MODEL_COL_CANDS  = ["model", "alias", "model_alias", "price_key", "engine"]

# 只画 child（可选）
CHILD_ONLY = False

# MDS 超参（与 TSP/SR 对齐）
MDS_MAX_POINTS   = 4000
PER_BUCKET       = 60
MDS_KW = dict(
    n_components=2,
    dissimilarity="precomputed",
    n_init=1,
    max_iter=300,
    eps=1e-3,
    random_state=42,
    verbose=1,
)

# OOS 插值：KNN & 权重
OOS_K = 8
OOS_P = 2.0     # 权重=1/(d+1e-8)^P
RNG = np.random.default_rng(42)

# 预计算距离矩阵缓存（可选）
USE_PRECOMPUTED = True
DIST_CACHE_DIR = Path(".obp_mds_dist_cache")     # 👈 新增：OBP 的距离缓存目录
DIST_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ========== 通用工具 ==========
def find_col(df: pd.DataFrame, cands: List[str], required: bool=True) -> Optional[str]:
    for c in cands:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Required column not found. Tried: {cands}")
    return None

def load_index(index_path: str) -> pd.DataFrame:
    idx = pd.read_csv(index_path)
    path_col = find_col(idx, PATH_COL_CANDS, required=True)
    keep = [path_col]
    # 把 index 里跟模型有关的列也带上（用于兜底）
    for c in MODEL_COL_CANDS:
        if c in idx.columns: keep.append(c)
    out = idx[keep].copy()
    out.rename(columns={path_col: "csv_file_path"}, inplace=True)
    return out

def read_csv_maybe(path: str) -> Optional[pd.DataFrame]:
    p = Path(path)
    if not p.exists():
        print(f"[warn] missing CSV: {p}")
        return None
    try:
        return pd.read_csv(p)
    except Exception as e:
        print(f"[warn] read failed: {p} -> {e}")
        return None

def robust_minmax(x: np.ndarray, q=(1, 99)):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return x
    lo, hi = np.nanpercentile(x, q)
    den = (hi - lo) if hi > lo else 1.0
    y = (x - lo) / den
    return 1 - np.clip(y, 0, 1)      

def stratified_sample_idx(df: pd.DataFrame, per_bucket=PER_BUCKET, total_cap=MDS_MAX_POINTS) -> np.ndarray:
    parts = []
    for (m, g), sub in df.groupby(["model","generation"], sort=False):
        if len(sub) > per_bucket:
            parts.append(sub.sample(per_bucket, random_state=int(RNG.integers(1<<31))))
        else:
            parts.append(sub)
    out = pd.concat(parts, ignore_index=False)
    if len(out) > total_cap:
        out = out.sample(total_cap, random_state=42)
    return out.index.to_numpy()


# ======= 余弦距离（1 - cos），加速型实现 =======
def l2_normalize_rows(A: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(A, axis=1, keepdims=True)
    n = np.where(n>eps, n, 1.0)
    return A / n

def cosine_D_fit(A_fit: np.ndarray) -> np.ndarray:
    """
    A_fit: (m, P) float32，已按行 L2 归一化
    返回 D_fit: (m, m) float32，D = 1 - (A_fit @ A_fit^T)（不取绝对值）
    """
    S = np.clip(A_fit @ A_fit.T, -1.0, 1.0).astype(np.float32)
    D = (1.0 - S).astype(np.float32)
    np.fill_diagonal(D, 0.0)
    return D

def cosine_D_query_to_fit(Aq: np.ndarray, Af: np.ndarray, block: int = 4000) -> np.ndarray:
    """
    Aq: (b, P)  Af: (m, P)  都需已 L2 归一化
    返回 Dq: (b, m) float32
    分块避免显存/内存峰值
    """
    b = Aq.shape[0]; m = Af.shape[0]
    out = np.zeros((b, m), dtype=np.float32)
    for s in range(0, b, block):
        sl = slice(s, min(b, s+block))
        S = np.clip(Aq[sl] @ Af.T, -1.0, 1.0).astype(np.float32)
        out[sl] = 1.0 - S
    return out

def oos_place_by_knn_cos(A_all, fit_idx, Y_fit, k=8, p=2.0, block=4000):
    """
    A_all: (N,P) 全体向量（未归一化/已归一化均可，本函数会归一化）
    fit_idx: (m,) 基点索引
    Y_fit: (m,2) MDS 结果
    返回 Y: (N,2)
    """
    N = A_all.shape[0]
    Y = np.zeros((N, 2), dtype=np.float32)
    Y[fit_idx] = Y_fit

    mask = np.ones(N, dtype=bool)
    mask[fit_idx] = False
    q_idx = np.where(mask)[0]
    if len(q_idx) == 0:
        return Y

    Af = l2_normalize_rows(A_all[fit_idx].astype(np.float32, copy=False))
    for s in range(0, len(q_idx), block):
        ids = q_idx[s: s+block]
        Aq  = l2_normalize_rows(A_all[ids].astype(np.float32, copy=False))
        Dq  = cosine_D_query_to_fit(Aq, Af, block=8000)        # (b, m)

        # 取每行的 k 近邻
        m = Af.shape[0]
        if k < m:
            nn = np.argpartition(Dq, kth=k, axis=1)[:, :k]
            rows = np.arange(len(ids))[:, None]
            dnn = Dq[rows, nn]
        else:
            nn = np.tile(np.arange(m), (len(ids),1))
            dnn = Dq

        w = 1.0 / (dnn + 1e-8) ** p
        w = w / w.sum(axis=1, keepdims=True)
        Y[ids] = (w[..., None] * Y_fit[nn]).sum(axis=1).astype(np.float32)

    return Y
# ========== Parquet 读取：从外部 parquet 还原每行 embedding ==========
def _guess_model_name(df: pd.DataFrame, csv_path: str, index_row: pd.Series) -> str:
    # 优先 CSV 内列，其次 index 行上的模型列，否则用文件名
    for c in MODEL_COL_CANDS:
        if c in df.columns and df[c].notna().any():
            return str(df[c].dropna().iloc[0])
        if c in index_row and isinstance(index_row[c], str) and index_row[c].strip():
            return str(index_row[c]).strip()
    return Path(csv_path).stem

def _hash_text_sha1(s: str) -> str:
    import hashlib
    return hashlib.sha1((s or "").encode("utf-8", errors="ignore")).hexdigest()

def _collect_parquet_candidates(csv_path: str, task_key: str) -> List[Path]:
    p = Path(csv_path)
    cands = []
    # 1) 同目录同名
    local = p.with_suffix(".parquet")
    if local.exists():
        cands.append(local)
    # 2) 根目录下按分区(task_key/model/…)存的
    root = Path(PARQUET_ROOT)
    if root.exists():
        # 尝试两种常见目录结构
        # a) write_to_dataset 分区：task_key=…/model=…/*.parquet
        patt1 = str(root / f"task_key={task_key}" / "model=*/**/*.parquet")
        cands += [Path(x) for x in glob.glob(patt1, recursive=True)]
        patt1b = str(root / f"task_key={task_key}" / "**/*.parquet")
        cands += [Path(x) for x in glob.glob(patt1b, recursive=True)]
        # b) 平铺
        patt2 = str(root / "**/*.parquet")
        cands += [Path(x) for x in glob.glob(patt2, recursive=True)]
    # 去重
    uniq = []
    seen = set()
    for q in cands:
        if q not in seen and q.exists():
            uniq.append(q); seen.add(q)
    return uniq

def _read_parquet_df(pq_path: Path) -> Optional[pd.DataFrame]:
    try:
        return pd.read_parquet(pq_path)
    except Exception:
        try:
            # 如果缺 pyarrow，就试试 fastparquet
            return pd.read_parquet(pq_path, engine="fastparquet")
        except Exception as e:
            print(f"[warn] failed reading parquet: {pq_path} -> {e}")
            return None
def _attach_embeddings(df_csv: pd.DataFrame,
                                    csv_path: str,
                                    task_key: str) -> Optional[pd.DataFrame]:
    """
    返回一个 DataFrame，包含：
      generation / fitness_raw / type / model / emb  (emb 为 np.ndarray)

    逻辑：
      - 找到与 csv 对应的 .parquet（同名或在 PARQUET_ROOT 下）
      - 兼容两类 schema：
         A) {"row_idx","generation","type","fitness_raw","source_csv","embedding", ...}
         B) {"row_id","generation","type","fitness","file_source","embedding", ...}
      - 优先按 row_idx/row_id 对齐；找不到则尝试同名文件 + 顺序对齐
      - **fitness_raw 优先用 parquet 里的 fitness/fitness_raw，
        如果没有则退回 CSV 里的 fitness 列**
    """
    # 必要列名（generation / type 从 CSV 里拿）
    gen_col  = find_col(df_csv, GEN_COL_CANDS, required=True)
    type_col = find_col(df_csv, TYPE_COL_CANDS, required=False)  # 允许缺失
    # 必要列名
    fit_col = find_col(df_csv, FITN_COL_CANDS, required=True)

    parquets = _collect_parquet_candidates(csv_path, task_key)
    if not parquets:
        print(f"[warn] no parquet found for: {csv_path}")
        return None

    parquets = _collect_parquet_candidates(csv_path, task_key)
    if not parquets:
        print(f"[warn] no parquet found for: {csv_path}")
        return None

    for pq_path in parquets:
        pdf = _read_parquet_df(pq_path)
        if pdf is None or pdf.empty:
            continue

        # ------- 统一书写 + 取 parquet 中的 fitness -------
        col_rowidx = None
        if "row_idx" in pdf.columns:
            col_rowidx = "row_idx"
        elif "row_id" in pdf.columns:
            col_rowidx = "row_id"

        if "fitness_raw" in pdf.columns:
            pdf["_fitness_from_pq"] = pd.to_numeric(pdf["fitness_raw"], errors="coerce")
        elif "fitness" in pdf.columns:
            pdf["_fitness_from_pq"] = pd.to_numeric(pdf["fitness"], errors="coerce")
        else:
            pdf["_fitness_from_pq"] = np.nan   # 如果缺失，再退回 CSV

        # ---------- 情况 1：通过 row_idx / row_id 精确对齐 ----------
        if col_rowidx is not None:
            tmp_csv = df_csv.reset_index(drop=False).rename(columns={"index": "row_id"})
            merged = tmp_csv.merge(pdf, on="row_id", how="left", suffixes=("","_pq"))

            if merged["embedding"].notna().any():
                # 优先用 parquet 的 fitness，如果全 NaN 再退回 CSV
                if merged["_fitness_from_pq"].notna().any():
                    fitness_raw = pd.to_numeric(merged["_fitness_from_pq"], errors="coerce")
                elif fit_csv_col is not None:
                    fitness_raw = pd.to_numeric(merged[fit_csv_col], errors="coerce")
                else:
                    # 没法得到 fitness，就跳过这个 parquet
                    continue

                out = pd.DataFrame({
                    "generation": pd.to_numeric(merged[gen_col], errors="coerce"),
                    "fitness_raw": fitness_raw,
                    "type": (merged[type_col].astype(str) if type_col in merged.columns else "unknown"),
                    "model": None,  # 后续在 build_obp_mds_cache 里再填
                    "emb": merged["embedding"].apply(
                        lambda v: np.asarray(v, np.float32) if isinstance(v, (list, np.ndarray)) else np.nan
                    ),
                })
                out = out.dropna(subset=["generation", "fitness_raw", "emb"])
                if not out.empty:
                    return out

        # ---------- 情况 2：fallback，同名 source_csv / file_source + 顺序对齐 ----------
        stem = Path(csv_path).stem
        if "source_csv" in pdf.columns:
            subpdf = pdf[pdf["source_csv"].astype(str).str.contains(stem, na=False)].copy()
        elif "file_source" in pdf.columns:
            subpdf = pdf[pdf["file_source"].astype(str).str.contains(stem, na=False)].copy()
        else:
            subpdf = pdf.copy()

        if not subpdf.empty and "embedding" in subpdf.columns:
            n = min(len(df_csv), len(subpdf))
            if n <= 0:
                continue

            # 还是优先 parquet 的 fitness
            if "_fitness_from_pq" in subpdf.columns and subpdf["_fitness_from_pq"].notna().any():
                fitness_raw = pd.to_numeric(subpdf["_fitness_from_pq"].iloc[:n], errors="coerce")
            elif fit_csv_col is not None:
                fitness_raw = pd.to_numeric(df_csv[fit_csv_col].iloc[:n], errors="coerce")
            else:
                continue

            out = pd.DataFrame({
                "generation": pd.to_numeric(df_csv[gen_col].iloc[:n], errors="coerce"),
                "fitness_raw": fitness_raw,
                "type": (df_csv[type_col].iloc[:n].astype(str) if type_col in df_csv.columns else "unknown"),
                "model": None,
                "emb": [
                    np.asarray(x, dtype=np.float32) if isinstance(x, (list, np.ndarray)) else np.nan
                    for x in subpdf["embedding"].iloc[:n].tolist()
                ],
            })
            out = out.dropna(subset=["generation","fitness_raw","emb"])
            if not out.empty:
                return out

    print(f"[warn] failed to attach embeddings from parquet for: {csv_path}")
    return None

def render_mds_panel(df, title):
    plt.figure(figsize=(3.2,3))
    ax = plt.gca()

    cmap = plt.get_cmap("viridis")
    gnorm = Normalize(df["generation"].min(), df["generation"].max())
    cols = cmap(gnorm(df["generation"]))

    sizes = 10 + 80 * df["fitness_normed"]

    ax.scatter(df["mds_x"], df["mds_y"], 
               s=sizes, c=cols,
               edgecolor="white", linewidth=0.3, alpha=0.9)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=9)

    buf = BytesIO()
    plt.tight_layout()
    plt.savefig(buf, format="png", dpi=120)
    plt.close()

    buf.seek(0)   # ←←← 关键补充
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def build_obp_mds_cache(INDEX_FILES: Dict[str,str], PARQUET_ROOT: str):
    OBP_MDS_cache = {}

    for task, index_csv in INDEX_FILES.items():
        print(f"\n=== OBP Subtask: {task} ===")

        idx = pd.read_csv(index_csv)
        if "csv_file_path" not in idx.columns:
            raise ValueError(f"{index_csv}缺少 csv_file_path 列")

        rows = []
        for _, row in idx.iterrows():
            p = Path(row["csv_file_path"])
            if not p.exists():
                print(f"[warn] missing {p}")
                continue

            df = pd.read_csv(p)
            if df.empty:
                continue

            out = attach_embedding(df, str(p), PARQUET_ROOT, task)
            print("ATTACH CHECK:", task, p.name, "→ returned rows:", 0 if out is None else len(out))
            if out is None:
                continue

            out["model"] = row["model"] if "model" in row else Path(p).stem
            rows.append(out[["generation","fitness_raw","emb","model"]])

        if not rows:
            print(f"[warn] no data for {task}")
            continue

        big = pd.concat(rows, ignore_index=True)
        big = big.reset_index(drop=True)
        print(f"[info] collected rows={len(big)}, models={big['model'].nunique()}")

        # —— MDS 主流程 ——
        lens = big["emb"].apply(lambda a: len(a) if isinstance(a, np.ndarray) else 0)
        if lens.eq(0).all():
            print(f"[warn] empty embeddings for {subtask}")
            continue
        mode_len = int(lens[lens>0].mode().iloc[0])    # 取众数长度
        big = big[lens.eq(mode_len)].copy().reset_index(drop=True)
        if big.empty:
            print(f"[warn] embeddings empty after length filtering for {subtask}")
            continue

        A = np.stack(big["emb"].tolist()).astype(np.float32)
        N, P = A.shape
        print(f"[info] embedding matrix={A.shape}")

        # 抽样
        if len(big) <= MDS_MAX_POINTS:
            fit_idx = np.arange(len(big))
        else:
            fit_idx = stratified_sample_idx(big, per_bucket=PER_BUCKET, total_cap=MDS_MAX_POINTS)

        Af = l2_normalize_rows(A[fit_idx])

        # ==== ⭐ 这里加距离矩阵缓存：一 task 一份 ====
        cache_key = f"{task}_m{len(fit_idx)}_p{P}_cos.npy"
        cache_path = DIST_CACHE_DIR / cache_key

        if USE_PRECOMPUTED and cache_path.exists():
            print(f"[mds] load cached D_fit: {cache_path.name}")
            D_fit = np.load(cache_path)
        else:
            print(f"[mds] computing D_fit on {len(fit_idx)} points (of {len(big)}) …")
            D_fit = cosine_D_fit(Af)
            if USE_PRECOMPUTED:
                try:
                    np.save(cache_path, D_fit)
                    print(f"[mds] cached: {cache_path.name}")
                except Exception as e:
                    print(f"[warn] cache save failed: {e}")

        Y_fit = MDS(**MDS_KW).fit_transform(D_fit)

        Y = oos_place_by_knn_cos(A, fit_idx, Y_fit, k=OOS_K, p=OOS_P,block = 4000)
        big["mds_x"], big["mds_y"] = Y[:,0], Y[:,1]
        big["fitness_normed"] = big["fitness_raw"]

        big["fitness_normed"] = robust_minmax(big["fitness_raw"].to_numpy(), q=(1,99))

        # 每个模型生成 panel
        for model, dfm in big.groupby("model"):
            key = (task, str(model))
            img = render_mds_panel(dfm, f"{task} | {model}")
            OBP_MDS_cache[key] = img

    return OBP_MDS_cache

In [ ]:
INDEX_FILES = {
    "binpacking_or3": "/Users/lievretre/Desktop/evo_eval_sheets/bin_packing_or3_raw.csv",
    "binpacking_weibull": "/Users/lievretre/Desktop/evo_eval_sheets/bin_packing_weibull_raw.csv",
}

PARQUET_ROOT = "parquet_embeddings"

OBP_MDS_cache = build_obp_mds_cache(INDEX_FILES, PARQUET_ROOT)

In [ ]:
import random
import base64
from io import BytesIO
from PIL import Image
from IPython.display import display

def test_one_panel(OBP_MDS_cache):
    """随机选一个 panel 解码并显示"""
    if not OBP_MDS_cache:
        print("[ERROR] OBP_MDS_cache is empty!")
        return
    
    # 随机挑选一个 key
    key = random.choice(list(OBP_MDS_cache.keys()))
    print(f"Testing panel: {key}")

    b64 = OBP_MDS_cache[key]
    if not isinstance(b64, str) or len(b64) < 100:
        print("[ERROR] Base64 string too short → probably broken")
        return
    
    try:
        img_data = base64.b64decode(b64)
        img = Image.open(BytesIO(img_data))
        display(img)
        print("[OK] Image decoded and displayed successfully.")
    except Exception as e:
        print("[ERROR] Failed to decode image:", e)

def test_all_panels(OBP_MDS_cache):
    """依次测试所有 panel（不会显示，只检查是否可以 decode）"""
    ok, bad = 0, 0
    for key, b64 in OBP_MDS_cache.items():
        try:
            img_data = base64.b64decode(b64)
            Image.open(BytesIO(img_data))
            ok += 1
        except:
            print("[BROKEN]", key)
            bad += 1

    print("\n=== SUMMARY ===")
    print("Valid images:", ok)
    print("Broken images:", bad)

In [ ]:
import math
import base64
from io import BytesIO
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize


def robust_minmax_reverse(x, q=(0, 100)):
    """
    全局鲁棒 min-max，且方向反过来：
    原始值越小 → 数值越大（适用于“越小越好”的 fitness）
    """
    x_np = np.asarray(x, float)
    if x_np.size == 0:
        return x_np
    lo, hi = np.nanpercentile(x_np, q)
    den = (hi - lo) if hi > lo else 1.0
    return np.clip((hi - x_np) / den, 0, 1)


def _render_promptopt_panel(df, title, gen_norm, cmap):
    """
    画单个 (subtask, model) 面板，返回 base64 PNG
    风格尽量复刻你之前的 promptopt 可视化：
      - 背景 hexbin
      - 前景散点：颜色=generation，大小=fitness_normed
    """
    fig, ax = plt.subplots(figsize=(3.2, 2.8))

    # 颜色与尺寸
    cols = cmap(gen_norm(df["generation"].to_numpy(float)))
    sizes = 10 + 80 * df["fitness_normed"].to_numpy(float)

    # 背景密度（hexbin）
    try:
        ax.hexbin(df["mds_x"], df["mds_y"],
                  gridsize=35, cmap="Greys", mincnt=3,
                  alpha=0.15, zorder=1)
    except Exception:
        # 真出错就直接跳过 hexbin
        pass

    # 散点
    ax.scatter(df["mds_x"], df["mds_y"],
               s=sizes, c=cols,
               edgecolor="white", linewidth=0.35,
               alpha=0.92, zorder=2)

    ax.set_title(title, fontsize=10, pad=4)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)

    buf = BytesIO()
    plt.tight_layout()
    plt.savefig(buf, format="png", dpi=160)
    plt.close(fig)

    buf.seek(0)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def build_promptopt_mds_cache(TASK_PARQUETS: Dict[str, Path]) -> Dict[Tuple[str, str], str]:
    """
    输入：
      TASK_PARQUETS = {
          "Summarization": Path("...promptopt_sum_cos_coords.parquet"),
          "Simplification": Path("...promptopt_sim_cos_coords.parquet"),
      }

    parquet 内至少需要列：
      - generation
      - fitness  （原始值：越小越好）
      - mds_x, mds_y
      - model

    输出：
      PROMPTOPT_MDS_cache[(subtask, model)] = <base64 PNG>
    """

    dfs = []
    for subtask, p in TASK_PARQUETS.items():
        p = Path(p)
        if not p.exists():
            print(f"[warn] missing parquet: {p}")
            continue
        try:
            df = pd.read_parquet(p)
        except Exception as e:
            print(f"[error] failed reading {p}: {e}")
            continue

        need_cols = ["generation", "fitness", "mds_x", "mds_y", "model"]
        miss = [c for c in need_cols if c not in df.columns]
        if miss:
            print(f"[warn] {p} missing columns: {miss}, skip.")
            continue

        df = df.copy()
        df["subtask"] = subtask
        dfs.append(df)

    if not dfs:
        raise ValueError("No promptopt parquet loaded; please check TASK_PARQUETS paths.")

    all_df = pd.concat(dfs, ignore_index=True)

    # ===== 统一归一：generation + fitness（跨两个任务全局 pool）=====
    gen_norm = Normalize(vmin=all_df["generation"].min(),
                         vmax=all_df["generation"].max())
    all_df["fitness_normed"] = robust_minmax_reverse(all_df["fitness"])

    # ===== 按 (subtask, model) 构建 cache =====
    PROMPTOPT_MDS_cache: Dict[Tuple[str, str], str] = {}
    cmap = plt.get_cmap("viridis")

    for (subtask, model_name), sub in all_df.groupby(["subtask", "model"]):
        sub = sub.copy().sort_values("generation")

        # 标题里的 model 显示做一点清理（跟你之前脚本一样）
        title_model = str(model_name)
        title_model = title_model.replace("_np2_nc5_simple", "") \
                                 .replace(":free", "") \
                                 .replace("-instruct", "")
        idx = title_model.rfind("_")
        if idx != -1:
            title_model = title_model[idx+1:]

        title = f"{subtask} | {title_model}"

        img_b64 = _render_promptopt_panel(sub, title, gen_norm, cmap)
        PROMPTOPT_MDS_cache[(subtask, str(model_name))] = img_b64

    print(f"[info] built promptopt MDS cache with {len(PROMPTOPT_MDS_cache)} panels.")
    return PROMPTOPT_MDS_cache

In [ ]:
TASK_PARQUETS = {
    "Summarization": Path("/Users/lievretre/Desktop/mds_pack/promptopt_sum_cos_coords.parquet"),
    "Simplification": Path("/Users/lievretre/Desktop/mds_pack/promptopt_sim_cos_coords.parquet"),
}

PROMPTOPT_MDS_cache = build_promptopt_mds_cache(TASK_PARQUETS)

In [ ]:
print(sorted(scatter_cache.keys()))